# PE2 · Motor LLM base — ejecución en Colab

Antes de empezar: en el panel **🔑 Secrets** (izquierda) agregá un secreto llamado `GEMINI_API_KEY` (o `GOOGLE_API_KEY`) con tu clave de [Google AI Studio](https://aistudio.google.com/apikey) y activá *Acceso al notebook*.

Corré las celdas en orden.

## 1. Subir el proyecto
Subí el archivo `fde-medidas-catalogo.zip`. (Cuando el repo esté en GitHub, se reemplaza por `!git clone <url>`.)

In [ ]:
from google.colab import files
import os, zipfile
subido = files.upload()
zipfile.ZipFile(next(iter(subido))).extractall('/content')
os.chdir('/content/fde-medidas-catalogo')
!ls -R | head -30

## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

## 3. Cargar la API key desde Secrets
La clave nunca queda escrita en el notebook ni en el repositorio.

In [ ]:
from google.colab import userdata
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['PROVEEDOR'] = 'gemini'
os.environ['MODELO'] = 'gemini-3.1-flash-lite'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.makedirs('resultados', exist_ok=True)
print('Clave cargada:', bool(os.environ['GEMINI_API_KEY']))

## 4. Medición de las 5 consultas (tabla con promedios)

In [ ]:
!python -m src.medir 2>&1 | tee resultados/terminal_medir.txt

## 5. Una ejecución exitosa

In [ ]:
!python -m src.motor --sku SKU-10231 --descripcion "Galletitas dulces 350 g" --subcategoria snacks --texto "Caja 210 x 60 x 45 mm, peso neto 350 grs" 2>&1 | tee resultados/terminal_ok.txt

## 6. Una ejecución con error (entrada inválida, a propósito)

In [ ]:
!python -m src.motor --sku 10231 --descripcion "Ga" --subcategoria golosinas --texto "Caja 120x80x50 mm" 2>&1 | tee resultados/terminal_error.txt

## 7. Descargar los resultados
Baja `resultados.zip` con la tabla y las salidas de terminal para el PDF.

In [ ]:
!zip -qr resultados.zip resultados
files.download('resultados.zip')